In [1]:
import pandas as pd
import re
from dateutil import parser
from sqlalchemy import create_engine

year = 2022
df = pd.read_csv('../amazon_india_2022.csv', low_memory=False)

def clean_price(val):
    val = str(val).replace('₹', '').replace(',', '').strip()
    try: return float(val)
    except: return None

def clean_rating(val):
    if pd.isna(val): return None
    val = str(val).lower().strip()
    if 'star' in val: return float(re.search(r'\d+\.?\d*', val).group())
    if '/' in val: a, b = val.split('/'); return round(float(a)/float(b)*5, 2)
    try: return float(val)
    except: return None

def clean_date(val):
    try: return parser.parse(val, dayfirst=True).strftime('%Y-%m-%d')
    except: return None

def clean_delivery(val):
    val = str(val).strip().lower()
    if val == 'same day': return 0
    if val == 'express': return 1
    if '-' in val:
        try:
            a, b = val.split('-')
            return round((float(a) + float(b.split()[0])) / 2)
        except: return None
    try:
        val = int(float(val))
        if val < 0: return None
        if val > 30: return None
        return val
    except: return None


true_values = ['True', 'TRUE', '1', 'Yes']
manual_mapping = {'Bombay': 'Mumbai', 'Madras': 'Chennai', 'Bengaluru': 'Bangalore', 'Calcutta': 'Kolkata'}
category_mapping = {'ELECTRONICS': 'Electronics', 'Electronics & Accessories': 'Electronics', 'Electronic': 'Electronics', 'Electronicss': 'Electronics'}
payment_mapping = {
    'Cash on Delivery': 'COD', 'C.O.D': 'COD',
    'CREDIT CARD': 'Credit Card', 'CC': 'Credit Card',
    'DEBIT CARD': 'Debit Card',
    'NET BANKING': 'Net Banking',
    'UPI': 'UPI', 'PhonePe': 'UPI', 'GooglePay': 'UPI', 'GPay': 'UPI'
}

df['original_price_inr'] = df['original_price_inr'].apply(clean_price)
df['product_rating'] = df['product_rating'].apply(clean_rating)
df['product_rating'] = df.groupby('category')['product_rating'].transform(lambda x: x.fillna(x.median()))
df['product_rating'] = df['product_rating'].fillna(df['product_rating'].median())
df['customer_rating'] = df['customer_rating'].apply(clean_rating)
df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())
df['order_date'] = df['order_date'].apply(clean_date)
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_days'] = df['delivery_days'].apply(clean_delivery)
df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median())
df['customer_age_group'] = df['customer_age_group'].fillna(df['customer_age_group'].mode()[0])
df['delivery_charges'] = df['delivery_charges'].fillna(0)
df['festival_name'] = df['festival_name'].fillna('No Festival')
df['is_prime_member'] = False
df['is_prime_eligible'] = df['is_prime_eligible'].isin(true_values)
df['is_festival_sale'] = df['is_festival_sale'].isin(true_values)
df['customer_city'] = df['customer_city'].str.strip().str.title().replace(manual_mapping)
df['category'] = df['category'].replace(category_mapping)
df['payment_method'] = df['payment_method'].replace(payment_mapping)

df['original_price_inr'] = df.apply(
    lambda row: row['discounted_price_inr'] / (1 - row['discount_percent']/100)
    if pd.isna(row['original_price_inr']) or row['original_price_inr'] > row['discounted_price_inr'] * 10
    else row['original_price_inr'], axis=1
)
df.loc[df['original_price_inr'] < df['discounted_price_inr'], 'original_price_inr'] = df.apply(
    lambda row: row['discounted_price_inr'] / (1 - row['discount_percent']/100)
    if row['original_price_inr'] < row['discounted_price_inr']
    else row['original_price_inr'], axis=1
)

df = df.drop_duplicates(subset=['customer_id', 'product_id', 'order_date', 'final_amount_inr'], keep='first')

print(df.isnull().sum())

df.to_csv(f'2022_amazon.csv', index=False)
engine = create_engine('mysql+pymysql://root:@127.0.0.1:3306/amazon_db?unix_socket=/Applications/XAMPP/xamppfiles/var/mysql/mysql.sock')
df.to_sql(f'transactions_{year}', engine, if_exists='replace', index=False)


transaction_id            0
order_date                0
customer_id               0
product_id                0
product_name              0
category                  0
subcategory               0
brand                     0
original_price_inr        0
discount_percent          0
discounted_price_inr      0
quantity                  0
subtotal_inr              0
delivery_charges          0
final_amount_inr          0
customer_city             0
customer_state            0
customer_tier             0
customer_spending_tier    0
customer_age_group        0
payment_method            0
delivery_days             0
delivery_type             0
is_prime_member           0
is_festival_sale          0
festival_name             0
customer_rating           0
return_status             0
order_month               0
order_year                0
order_quarter             0
product_weight_kg         0
is_prime_eligible         0
product_rating            0
dtype: int64


132000

In [4]:
df.customer_city.unique()


array(['Vadodara', 'Chennai', 'Delhi', 'Gorakhpur', 'Ludhiana', 'Pune',
       'Lucknow', 'Kanpur', 'Indore', 'Ahmedabad', 'Kolkata', 'Mumbai',
       'Bangalore', 'Coimbatore', 'Nagpur', 'Moradabad', 'Jaipur',
       'Visakhapatnam', 'Saharanpur', 'Patna', 'Chandigarh', 'Allahabad',
       'Bhubaneswar', 'Kochi', 'Hyderabad', 'Aligarh', 'Surat',
       'Bareilly', 'New Delhi', 'Varanasi', 'Meerut', 'Chenai',
       'Delhi Ncr', 'Mumba', 'Bengalore', 'Banglore'], dtype=object)